In [ ]:
import subprocess, sys, time
script = '\nimport time, sys\nimport torch, timm\nimport torch_xla\nimport torch_xla.runtime as xr\n\ndef fn(index):\n    dev = torch_xla.device()\n    rank, ws = xr.global_ordinal(), xr.world_size()\n    model = timm.create_model("efficientnet_b0", pretrained=False, num_classes=12).to(dev)\n    opt = torch.optim.Adam(model.parameters(), 3e-4)\n    B = 240  # 8 studies x 6 slots x 5 groups\n    x = torch.randn(B, 3, 224, 224, device=dev)\n    y = (torch.rand(B, 12, device=dev) > 0.5).float()\n    amp = True\n    times = []\n    for step in range(25):\n        t = time.time()\n        try:\n            with torch.autocast("xla", dtype=torch.bfloat16, enabled=amp):\n                out = model(x)\n        except Exception as e:\n            print(rank, "autocast failed", type(e).__name__, str(e)[:150], flush=True); amp = False\n            out = model(x)\n        loss = torch.nn.functional.binary_cross_entropy_with_logits(out.float(), y)\n        loss.backward(); opt.step(); opt.zero_grad(); torch_xla.sync()\n        times.append(time.time() - t)\n    steady = sorted(times[5:])\n    print(f"rank {rank}/{ws} dev {dev} amp {amp} first {times[0]:.1f}s steady-median {steady[len(steady)//2]:.3f}s "\n          f"-> {B/steady[len(steady)//2]:.0f} img/s", flush=True)\n\nif __name__ == "__main__":\n    torch_xla.launch(fn)\n'
open('/kaggle/working/probe2.py', 'w').write(script)
t = time.time()
p = subprocess.run([sys.executable, '/kaggle/working/probe2.py'], capture_output=True, text=True, timeout=1500)
print('exit', p.returncode, 'wall', round(time.time() - t), 's')
print('\n'.join(l for l in p.stdout.splitlines() if l.startswith('rank') or 'failed' in l))
print('--- stderr tail ---'); print(p.stderr[-2500:])
